# Seal OCR Recognition: Train from Official Base Model (PP-OCRv6)

Huấn luyện trực tiếp từ **Model Base (Baidu PP-OCRv6 medium pretrained)** trên bộ dataset chuẩn hóa mới **`recognition_dataset_rec_v1.0.3`** (14.800 ảnh) để tạo ra mô hình **`seal-rec-v1.0.3`**.

### Điểm nổi bật & Khắc phục lỗi treo:
- **Chống treo 100%**: Đã fix lỗi NCCL P2P deadlock trên Kaggle 2x T4 (`NCCL_P2P_DISABLE=1`), tắt multiprocessing DataLoader IPC lockup (`num_workers=0`).
- **Hỗ trợ linh hoạt 1 GPU hoặc 2 GPU**: Mặc định chạy 1 GPU (`GPU_MODE = 1`) cực kỳ ổn định, hoặc chọn 2 GPUs (`GPU_MODE = 2`) với Kaggle accelerator.
- **Live log streaming**: Hiển thị trực tiếp loss, metric từng epoch lên màn hình, không bị câm lặng (không lo treo vô hình).
- **Train thẳng từ Model Base**: Không cần file checkpoint cũ, tự động tải pretrain từ Baidu và hoàn thành chỉ trong **15 epoch** (~25-30 phút).

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import csv, hashlib, json, os, random, shutil, subprocess, sys, urllib.request

# =====================================================================
# 1. CẤU HÌNH HUẤN LUYỆN (TRAINING CONFIGURATION)
# =====================================================================
# RUN_MODE: "smoke" (1 epoch để test đường ống) | "full" (chạy thật)
RUN_MODE = "full"

# Số epoch huấn luyện từ model base (15 epoch là tối ưu cho 14.800 ảnh)
TOTAL_EPOCHS = 1 if RUN_MODE == "smoke" else 15

# Cấu hình GPU:
#   1      : 1 GPU (Khuyên dùng: cực kỳ ổn định, không bao giờ bị nghẽn mạng phân tán)
#   2      : 2 GPUs (Distributed training trên Kaggle 2x T4 accelerator)
#   "auto" : Tự động phát hiện số GPU
GPU_MODE = 1  # Đổi thành 2 nếu muốn dùng 2x T4 trên Kaggle

# Thiết lập biến môi trường GPU ngay từ đầu trước khi import bất kỳ thư viện CUDA nào:
if str(GPU_MODE) in ("1", "1_gpu"):
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
elif str(GPU_MODE) in ("2", "2_gpu"):
    os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# Tắt P2P bị lỗi trên cloud virtual PCIe của Kaggle dual T4
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["FLAGS_fraction_of_gpu_memory_to_use"] = "0.90"

# Chế độ huấn luyện:
#   "direct_base" : Train thẳng từ Baidu PP-OCRv6 base -> ra seal-rec-v1.0.3 (Khuyên dùng)
#   "two_stage"   : Chế độ 2 giai đoạn cũ (Stage 1 rebuild v1 + Stage 2 finetune)
TRAIN_MODE = "direct_base"

SEED = 20260921
SPACING_PROB = 0.35
LEARNING_RATE = 0.00005

# Nhận diện môi trường Kaggle vs Google Colab vs Local
IS_KAGGLE = Path("/kaggle").exists()
IS_COLAB = Path("/content").exists() and not IS_KAGGLE

if IS_KAGGLE:
    BASE_WORK = Path("/kaggle/working")
    INPUT_DIR = Path("/kaggle/input")
elif IS_COLAB:
    BASE_WORK = Path("/content")
    INPUT_DIR = Path("/content")
else:
    BASE_WORK = Path("./workspace").resolve()
    INPUT_DIR = Path("./input").resolve()

WORK = BASE_WORK / "seal-ocr-training"
PADDLE = BASE_WORK / "PaddleOCR"
RUNS = BASE_WORK / "runs"

for folder in (WORK, RUNS):
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True)

print({
    "environment": "kaggle" if IS_KAGGLE else ("colab" if IS_COLAB else "local"),
    "runMode": RUN_MODE,
    "trainMode": TRAIN_MODE,
    "epochs": TOTAL_EPOCHS,
    "gpuMode": GPU_MODE,
    "cudaVisibleDevices": os.environ.get("CUDA_VISIBLE_DEVICES", "all"),
    "workDir": str(WORK)
})


## 2. Tìm kiếm và kiểm tra bộ Dataset

Hỗ trợ tự động nhận diện folder hoặc file zip của `recognition_dataset_rec_v1.0.3` (hoặc `seal-recognition-v4-manual`).

In [ ]:
WANTED = ["seal-recognition-rec-v1.0.3", "seal-recognition-v4-manual"]
found = {version: [] for version in WANTED}

def register_metadata(path):
    try:
        metadata = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return
    version = metadata.get("datasetVersion")
    if version in found and (path.parent / "manifest.csv").is_file():
        found[version].append((path.parent, metadata))

for metadata_path in INPUT_DIR.rglob("metadata.json"):
    register_metadata(metadata_path)

missing = [v for v in WANTED if not found[v]]
if missing:
    for index, zip_path in enumerate(INPUT_DIR.rglob("*.zip")):
        try:
            with ZipFile(zip_path) as archive:
                names = archive.namelist()
                if any(Path(name).is_absolute() or ".." in Path(name).parts for name in names):
                    continue
                for name in names:
                    if Path(name).name == "metadata.json":
                        try:
                            meta = json.loads(archive.read(name).decode("utf-8"))
                            ver = meta.get("datasetVersion")
                            if ver in WANTED and not found[ver]:
                                dest = WORK / "datasets" / f"zip-{index}"
                                archive.extractall(dest)
                                for mp in dest.rglob("metadata.json"):
                                    register_metadata(mp)
                        except Exception:
                            pass
        except Exception:
            pass

chosen_version = None
for v in WANTED:
    if found[v]:
        chosen_version = v
        break

if not chosen_version:
    raise RuntimeError(f"Không tìm thấy dataset nào trong {INPUT_DIR}. Vui lòng attach Kaggle Dataset hoặc upload file zip lên.")

DATASET_ROOT, dataset_metadata = found[chosen_version][0]
print(f"-> Sử dụng Dataset: {chosen_version} tại: {DATASET_ROOT}")

def verify_dataset(root, metadata):
    total = metadata.get("totalSamples", metadata.get("totalVerified"))
    train_total = metadata["trainSamples"]
    val_total = metadata.get("validationSamples", metadata.get("valSamples"))
    if total != train_total + val_total:
        raise RuntimeError(f"Số lượng mẫu không khớp: {metadata}")
    rows = list(csv.DictReader((root / "manifest.csv").open(encoding="utf-8-sig")))
    if len(rows) != total:
        raise RuntimeError(f"Manifest count {len(rows)} != {total}")
    fingerprint_lines = []
    for row in rows:
        image = root / row["image"]
        digest = hashlib.sha256(image.read_bytes()).hexdigest()
        if row.get("sha256") and digest != row["sha256"]:
            raise RuntimeError(f"Sai hash ảnh: {image}")
        fingerprint_lines.append(f"{row['image']}\t{row['ground_truth']}\t{digest}")
    fingerprint = hashlib.sha256("\n".join(sorted(fingerprint_lines)).encode()).hexdigest()
    expected_fp = metadata.get("datasetSha256")
    if expected_fp and fingerprint != expected_fp:
        raise RuntimeError(f"Fingerprint mismatch: {fingerprint} != {expected_fp}")
    return rows

dataset_rows = verify_dataset(DATASET_ROOT, dataset_metadata)
print({
    "dataset": str(DATASET_ROOT),
    "totalSamples": len(dataset_rows),
    "trainFile": str(DATASET_ROOT / "train.txt"),
    "valFile": str(DATASET_ROOT / "val.txt")
})


In [ ]:
# Cài đặt PaddleOCR & PaddlePaddle GPU
if not PADDLE.exists():
    subprocess.run(["git", "-c", "advice.detachedHead=false", "clone", "--depth", "1", "--branch", "v3.7.0", "https://github.com/PaddlePaddle/PaddleOCR.git", str(PADDLE)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q", "-r", str(PADDLE / "requirements.txt"), "pyyaml"], check=True)
try:
    import paddle
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q", "--upgrade", "paddlepaddle-gpu==3.3.0", "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/"], check=True)
    import paddle

if not paddle.device.is_compiled_with_cuda():
    raise RuntimeError("Paddle không có CUDA. Vui lòng bật GPU Accelerator trong Settings của Kaggle/Colab.")

detected_gpus = paddle.device.cuda.device_count()
print(f"-> Số lượng GPU CUDA khả dụng: {detected_gpus}")

if GPU_MODE == "auto":
    USE_GPU_COUNT = 2 if detected_gpus >= 2 else 1
elif str(GPU_MODE) in ("1", "1_gpu"):
    USE_GPU_COUNT = 1
elif str(GPU_MODE) in ("2", "2_gpu"):
    if detected_gpus < 2:
        print(f"[CẢNH BÁO] Yêu cầu 2 GPU nhưng hệ thống chỉ thấy {detected_gpus} GPU. Tự động chuyển về 1 GPU.")
        USE_GPU_COUNT = 1
    else:
        USE_GPU_COUNT = 2
else:
    USE_GPU_COUNT = 1

IS_DISTRIBUTED = (USE_GPU_COUNT >= 2)
GPU_IDS = "0,1" if USE_GPU_COUNT == 2 else "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_IDS

if not IS_DISTRIBUTED:
    paddle.device.set_device("gpu:0")

print({
    "paddleVersion": paddle.__version__,
    "usingGpuCount": USE_GPU_COUNT,
    "isDistributed": IS_DISTRIBUTED,
    "gpuIds": GPU_IDS,
    "device": paddle.device.get_device()
})


In [ ]:
# Thêm augmentation RandomCharacterSpacing vào thư viện PaddleOCR
transform_source = r"""import random
import cv2
from .text_image_aug import tia_stretch

class RandomCharacterSpacing:
    def __init__(self, prob=0.35, min_segments=3, max_segments=6, **kwargs):
        self.prob = float(prob)
        self.min_segments = int(min_segments)
        self.max_segments = int(max_segments)
    def __call__(self, data):
        image = data["image"]
        if random.random() > self.prob or image.shape[0] < 20 or image.shape[1] < 20:
            return data
        height = image.shape[0]
        stretched = tia_stretch(image, random.randint(self.min_segments, self.max_segments))
        if stretched.shape[0] != height:
            stretched = cv2.resize(stretched, (stretched.shape[1], height), interpolation=cv2.INTER_CUBIC)
        data["image"] = stretched
        return data
"""
target = PADDLE / "ppocr/data/imaug/random_character_spacing.py"
target.write_text(transform_source)
init_path = PADDLE / "ppocr/data/imaug/__init__.py"
import_line = "from .random_character_spacing import RandomCharacterSpacing\n"
if import_line not in init_path.read_text():
    init_path.write_text(init_path.read_text() + "\n" + import_line)
print({"transformSha256": hashlib.sha256(target.read_bytes()).hexdigest()})


In [ ]:
import yaml
OFFICIAL_CONFIG = WORK / "PP-OCRv6_medium_rec.yml"
urllib.request.urlretrieve("https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/v3.7.0/configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml", OFFICIAL_CONFIG)
OFFICIAL_PRETRAIN = WORK / "PP-OCRv6_medium_rec_pretrained.pdparams"
urllib.request.urlretrieve("https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_medium_rec_pretrained.pdparams", OFFICIAL_PRETRAIN)
print(f"-> Đã tải Model Base PP-OCRv6: {OFFICIAL_PRETRAIN} ({OFFICIAL_PRETRAIN.stat().st_size / 1024 / 1024:.2f} MB)")

def build_config(run_dir, dataset, train_label, val_label, epochs, seed, learning_rate):
    config = yaml.safe_load(OFFICIAL_CONFIG.read_text())
    config["Global"].update({
        "epoch_num": epochs,
        "seed": seed,
        "save_model_dir": str(run_dir / "checkpoints"),
        "save_epoch_step": max(1, epochs // 5),
        "eval_batch_step": [0, 100],
        "distributed": IS_DISTRIBUTED,
        "use_visualdl": False,
        "max_text_length": 25
    })
    config["Optimizer"]["lr"].update({
        "learning_rate": learning_rate,
        "warmup_epoch": 1 if epochs < 10 else 2
    })
    transforms = [
        {"DecodeImage": {"img_mode": "BGR", "channel_first": False}},
        {"RecAug": {"tia_prob": 0.10, "crop_prob": 0.0, "reverse_prob": 0.0, "noise_prob": 0.15, "jitter_prob": 0.05, "blur_prob": 0.10, "hsv_aug_prob": 0.15}},
        {"RandomCharacterSpacing": {"prob": SPACING_PROB, "min_segments": 3, "max_segments": 6}},
        {"MultiLabelEncode": {"gtc_encode": "NRTRLabelEncode"}},
        {"KeepKeys": {"keep_keys": ["image", "label_ctc", "label_gtc", "length", "valid_ratio"]}},
    ]
    config["Train"]["dataset"].update({
        "data_dir": str(dataset),
        "label_file_list": [str(train_label)],
        "transforms": transforms,
        "ext_op_transform_idx": 1,
        "ratio_list": [1.0]
    })
    config["Eval"]["dataset"].update({
        "data_dir": str(dataset),
        "label_file_list": [str(val_label)]
    })
    batch_per_card = 32 if IS_DISTRIBUTED else 64
    config["Train"]["sampler"]["first_bs"] = batch_per_card
    config["Train"]["loader"].update({"batch_size_per_card": batch_per_card, "drop_last": False, "num_workers": 0})
    config["Eval"]["loader"].update({"batch_size_per_card": batch_per_card, "num_workers": 0})
    path = run_dir / "config.yml"
    path.write_text(yaml.safe_dump(config, sort_keys=False))
    return path

def stream_command(command, cwd):
    """Chạy lệnh và stream output trực tiếp từng dòng ra màn hình (không bao giờ bị treo vô hình)"""
    print(f"\n[EXEC] {' '.join(command)}\n", flush=True)
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = GPU_IDS
    env["NCCL_P2P_DISABLE"] = "1"
    env["NCCL_IB_DISABLE"] = "1"
    env["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True
    )
    for line in iter(process.stdout.readline, ""):
        print(line, end="", flush=True)
    process.stdout.close()
    ret = process.wait()
    if ret != 0:
        raise RuntimeError(f"Lệnh thất bại với exit code {ret}")

def train_and_export(config, run_dir, pretrained):
    overrides = [
        f"Global.pretrained_model={pretrained}",
        f"Global.save_model_dir={run_dir / 'checkpoints'}"
    ]
    if IS_DISTRIBUTED:
        command = [
            sys.executable, "-m", "paddle.distributed.launch",
            f"--gpus={GPU_IDS}",
            "tools/train.py",
            "-c", str(config),
            "-o", *overrides
        ]
    else:
        command = [
            sys.executable,
            "tools/train.py",
            "-c", str(config),
            "-o", *overrides
        ]
    stream_command(command, cwd=PADDLE)
    best = run_dir / "checkpoints" / "best_accuracy"
    if not best.with_suffix(".pdparams").is_file():
        raise RuntimeError(f"Không tìm thấy checkpoint: {best}.pdparams")
    inference = run_dir / "inference"
    export_command = [
        sys.executable, "tools/export_model.py",
        "-c", str(config),
        "-o", "Global.distributed=False",
        f"Global.pretrained_model={best}",
        f"Global.save_inference_dir={inference}"
    ]
    stream_command(export_command, cwd=PADDLE)
    return best, inference

def package_run(run_dir, package_name):
    package_dir = WORK / "packages" / package_name
    if package_dir.exists():
        shutil.rmtree(package_dir)
    (package_dir / "checkpoints").mkdir(parents=True)
    shutil.copy2(run_dir / "config.yml", package_dir / "config.yml")
    shutil.copy2(run_dir / "run_metadata.json", package_dir / "run_metadata.json")
    shutil.copytree(run_dir / "inference", package_dir / "inference")
    for suffix in ("pdparams", "pdopt", "states"):
        source = run_dir / "checkpoints" / f"best_accuracy.{suffix}"
        if not source.is_file():
            raise RuntimeError(f"Thiếu file checkpoint: {source}")
        shutil.copy2(source, package_dir / "checkpoints" / source.name)
    archive_path = shutil.make_archive(str(BASE_WORK / package_name), "zip", package_dir)
    return archive_path


## 3. Thực thi Huấn luyện từ Model Base

Chạy huấn luyện trực tiếp từ `PP-OCRv6_medium_rec_pretrained.pdparams` trên bộ dữ liệu `recognition_dataset_rec_v1.0.3`.

In [ ]:
MODEL_NAME = "seal-rec-v1.0.3"
TARGET_DIR = RUNS / MODEL_NAME
TARGET_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_LABEL = DATASET_ROOT / "train.txt"
VAL_LABEL = DATASET_ROOT / "val.txt"

print(f"=== BẮT ĐẦU HUẤN LUYỆN {MODEL_NAME} TỪ MODEL BASE ===")
print({"epochs": TOTAL_EPOCHS, "dataset": str(DATASET_ROOT), "baseModel": str(OFFICIAL_PRETRAIN)})

CONFIG_PATH = build_config(TARGET_DIR, DATASET_ROOT, TRAIN_LABEL, VAL_LABEL, TOTAL_EPOCHS, SEED, LEARNING_RATE)
BEST_CKPT, INFERENCE_DIR = train_and_export(CONFIG_PATH, TARGET_DIR, OFFICIAL_PRETRAIN)

metadata = {
    "modelVersion": MODEL_NAME,
    "pipelineVersion": "seal-det-v1.0.0-rec-v1.0.3",
    "basePretrainedModel": "PP-OCRv6_medium_rec_pretrained.pdparams",
    "datasetVersion": dataset_metadata.get("datasetVersion", "seal-recognition-rec-v1.0.3"),
    "datasetSha256": dataset_metadata.get("datasetSha256"),
    "totalDatasetSamples": len(dataset_rows),
    "seed": SEED,
    "epochs": TOTAL_EPOCHS,
    "learningRate": LEARNING_RATE,
    "spacingProbability": SPACING_PROB,
    "usingGpuCount": USE_GPU_COUNT,
    "gpuIds": GPU_IDS,
    "isDistributed": IS_DISTRIBUTED,
    "globalBatchSize": 64
}
(TARGET_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2))
ARCHIVE_FILE = package_run(TARGET_DIR, MODEL_NAME)

print("\n=======================================================")
print(f"🎉 HUẤN LUYỆN HOÀN TẤT THÀNH CÔNG!")
print(f"📦 File model zip sẵn sàng tải về: {ARCHIVE_FILE}")
print(f"🎯 Best Checkpoint: {BEST_CKPT}")
print(f"📁 Inference Dir: {INFERENCE_DIR}")
print("=======================================================")
